In [1]:

import pandas as pd
import numpy as np
import time

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [2]:
#Serve para ler o arquivo CSV e transforma a coluna Data em data
df = pd.read_csv("../dados/mecaniqa_dataset.csv", parse_dates=["Data"])

df = df.sort_values("Data")
df = df.set_index("Data")

print("Tamanho dos dados:", df.shape)
print("Colunas:", df.columns.tolist())

display(df.head())


Tamanho dos dados: (731, 2)
Colunas: ['Trocas_Oleo', 'Manutencao_Motor']


,Trocas_Oleo,Manutencao_Motor
Data,,
2024-01-01,11.0,2.0
2024-01-02,9.0,4.0
2024-01-03,12.0,1.0
2024-01-04,15.0,8.0
2024-01-05,25.0,10.0


In [3]:

# Guarda uma cópia dos dados originais
dados = df.copy()
# Guarda a quantidade de trocas de óleo do dia anterior
dados["Lag_1"] = dados["Trocas_Oleo"].shift(1)
# Guarda a quantidade de trocas de óleo de 7 dias atrás
dados["Lag_7"] = dados["Trocas_Oleo"].shift(7)
# Calcula a média de trocas de óleo dos últimos 7 dias
dados["Media_7"] = dados["Trocas_Oleo"].shift(1).rolling(7).mean()
# Identifica o dia da semana
dados["Dia_Semana"] = dados.index.dayofweek

colunas = ["Lag_1", "Lag_7", "Media_7", "Dia_Semana"]
dados = dados.dropna(subset=colunas + ["Trocas_Oleo"])

X = dados[colunas]
y = dados["Trocas_Oleo"]

tscv = TimeSeriesSplit(n_splits=5)

print("Registros preparados:", len(dados))
print("Características:", colunas)
print("Divisões temporais:", tscv.n_splits)


Registros preparados: 711
Características: ['Lag_1', 'Lag_7', 'Media_7', 'Dia_Semana']
Divisões temporais: 5


In [4]:

# Modelos e valores que serão testados
modelos = {
    "Árvore de Decisão": (
        DecisionTreeRegressor(random_state=42),
        {
            "max_depth": [3, 5, 10, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    ),
    "Random Forest": (
        RandomForestRegressor(random_state=42),
        {
            "n_estimators": [50, 100, 200],
            "max_depth": [5, 10, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    )
}

resultados = []

for nome, (modelo, parametros) in modelos.items():
    print("\nTestando:", nome)

    # Grid Search
    grid = GridSearchCV(
        modelo,
        parametros,
        scoring="neg_mean_absolute_error",
        cv=tscv,
        n_jobs=-1
    )

    inicio = time.perf_counter()
    grid.fit(X, y)
    tempo_grid = time.perf_counter() - inicio

    resultados.append({
        "Modelo": nome,
        "Método": "Grid Search",
        "MAE": -grid.best_score_,
        "Tempo (s)": tempo_grid,
        "Melhores parâmetros": grid.best_params_
    })

    # Random Search
    random = RandomizedSearchCV(
        modelo,
        parametros,
        n_iter=10,
        scoring="neg_mean_absolute_error",
        cv=tscv,
        random_state=42,
        n_jobs=-1
    )

    inicio = time.perf_counter()
    random.fit(X, y)
    tempo_random = time.perf_counter() - inicio

    resultados.append({
        "Modelo": nome,
        "Método": "Random Search",
        "MAE": -random.best_score_,
        "Tempo (s)": tempo_random,
        "Melhores parâmetros": random.best_params_
    })

# Mostra a comparação dos resultados
tabela = pd.DataFrame(resultados)

display(
    tabela.sort_values("MAE")
    .reset_index(drop=True)
    .round({"MAE": 2, "Tempo (s)": 2})
)



Testando: Árvore de Decisão

Testando: Random Forest


,Modelo,Método,MAE,Tempo (s),Melhores parâmetros
0,Random Forest,Grid Search,4.44,175.06,"{'max_depth': 10, 'min_samples_leaf': 2, 'min_..."
1,Random Forest,Random Search,4.50,6.84,"{'n_estimators': 50, 'min_samples_split': 5, '..."
2,Árvore de Decisão,Grid Search,4.75,54.94,"{'max_depth': 5, 'min_samples_leaf': 1, 'min_s..."
3,Árvore de Decisão,Random Search,5.07,0.69,"{'min_samples_split': 5, 'min_samples_leaf': 4..."


## Análise de impacto e comparação de eficiência

In [5]:
from sklearn.model_selection import cross_val_score

# Comparar o Grid Search com o modelo inicial nas mesmas divisões temporais.
comparacao_impacto = []
comparacao_eficiencia = []

for nome, (modelo, _) in modelos.items():
    sem_ajuste = -cross_val_score(
        modelo, X, y,
        scoring="neg_mean_absolute_error",
        cv=TimeSeriesSplit(n_splits=5),
        n_jobs=-1
    ).mean()

    linha_grid = tabela[(tabela["Modelo"] == nome) & (tabela["Método"] == "Grid Search")].iloc[0]
    linha_random = tabela[(tabela["Modelo"] == nome) & (tabela["Método"] == "Random Search")].iloc[0]

    mae_grid = linha_grid["MAE"]
    ganho = (sem_ajuste - mae_grid) / sem_ajuste * 100
    print(f"\n{nome}")
    print("Melhores hiperparâmetros (Grid Search):", linha_grid["Melhores parâmetros"])
    print(f"MAE do modelo padrão: {sem_ajuste:.2f}")
    print(f"MAE com Grid Search: {mae_grid:.2f}")
    print(f"Variação do erro com o ajuste: {ganho:+.2f}% (positivo = redução do MAE)")

    comparacao_impacto.append({
        "Modelo": nome,
        "MAE padrão": sem_ajuste,
        "MAE tunado (Grid)": mae_grid,
        "Redução do MAE (%)": ganho
    })

    economia_tempo = linha_grid["Tempo (s)"] - linha_random["Tempo (s)"]
    reducao_tempo = 100 * economia_tempo / linha_grid["Tempo (s)"]
    diferenca_erro = linha_random["MAE"] - mae_grid
    comparacao_eficiencia.append({
        "Modelo": nome,
        "Grid (s)": linha_grid["Tempo (s)"],
        "Random (s)": linha_random["Tempo (s)"],
        "Economia de tempo (s)": economia_tempo,
        "Economia de tempo (%)": reducao_tempo,
        "Diferença de MAE (Random - Grid)": diferenca_erro
    })

print("\nImpacto da otimização:")
display(pd.DataFrame(comparacao_impacto).round(2))
print("\nEficiência — Grid Search x Random Search:")
display(pd.DataFrame(comparacao_eficiencia).round(2))



Árvore de Decisão
Melhores hiperparâmetros (Grid Search): {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}
MAE do modelo padrão: 5.40
MAE com Grid Search: 4.75
Variação do erro com o ajuste: +11.92% (positivo = redução do MAE)

Random Forest
Melhores hiperparâmetros (Grid Search): {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 50}
MAE do modelo padrão: 4.73
MAE com Grid Search: 4.44
Variação do erro com o ajuste: +6.03% (positivo = redução do MAE)

Impacto da otimização:


,Modelo,MAE padrão,MAE tunado (Grid),Redução do MAE (%)
0,Árvore de Decisão,5.40,4.75,11.92
1,Random Forest,4.73,4.44,6.03



Eficiência — Grid Search x Random Search:


,Modelo,Grid (s),Random (s),Economia de tempo (s),Economia de tempo (%),Diferença de MAE (Random - Grid)
0,Árvore de Decisão,54.94,0.69,54.24,98.74,0.32
1,Random Forest,175.06,6.84,168.22,96.09,0.06
